# Yash Gaikwad
# Roll no-381071

Implement a Federated Learning system where clients train local models on private data and 
share only noisy model updates with the server using differential privacy, so the server can build 
a global model without accessing raw data. 

In [1]:
pip install torch torchvision numpy

  Obtaining dependency information for torch from https://files.pythonhosted.org/packages/d1/bd/9912d30b68845256aabbb4a40aeefeef3c3b20db5211ccda653544ada4b6/torch-2.11.0-cp311-cp311-win_amd64.whl.metadata
  Obtaining dependency information for torchvision from https://files.pythonhosted.org/packages/10/58/ed8f7754299f3e91d6414b6dc09f62b3fa7c6e5d63dfe48d69ab81498a37/torchvision-0.26.0-cp311-cp311-win_amd64.whl.metadata
  Obtaining dependency information for numpy from https://files.pythonhosted.org/packages/bd/63/05d193dbb4b5eec1eca73822d80da98b511f8328ad4ae3ca4caf0f4db91d/numpy-2.4.4-cp311-cp311-win_amd64.whl.metadata
  Obtaining dependency information for filelock from https://files.pythonhosted.org/packages/3b/21/2f728888c45033d34a417bfcd248ea2564c9e08ab1bfd301377cf05d5586/filelock-3.28.0-py3-none-any.whl.metadata
  Obtaining dependency information for sympy>=1.13.3 from https://files.pythonhosted.org/packages/a2/09/77d55d46fd61b4a135c444fc97158ef34a095e5681d0a6c10b75bf356191/sympy-1

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 23.2.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import copy

# -----------------------------
# Simple Model
# -----------------------------
class SimpleModel(nn.Module):
    def __init__(self):
        super(SimpleModel, self).__init__()
        self.fc = nn.Linear(10, 2)

    def forward(self, x):
        return self.fc(x)

# -----------------------------
# Differential Privacy Functions
# -----------------------------
def clip_gradients(model, max_norm):
    total_norm = 0
    for p in model.parameters():
        if p.grad is not None:
            total_norm += p.grad.data.norm(2).item() ** 2
    total_norm = total_norm ** 0.5

    clip_coef = max_norm / (total_norm + 1e-6)
    if clip_coef < 1:
        for p in model.parameters():
            if p.grad is not None:
                p.grad.data.mul_(clip_coef)

def add_noise(model, noise_scale):
    for p in model.parameters():
        if p.grad is not None:
            noise = torch.normal(0, noise_scale, size=p.grad.shape)
            p.grad += noise

# -----------------------------
# Client Training
# -----------------------------
def train_client(model, data, targets, epochs=1, lr=0.01,
                 max_norm=1.0, noise_scale=0.1):
    
    model = copy.deepcopy(model)
    optimizer = optim.SGD(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    for _ in range(epochs):
        optimizer.zero_grad()
        outputs = model(data)
        loss = criterion(outputs, targets)
        loss.backward()

        # Apply Differential Privacy
        clip_gradients(model, max_norm)
        add_noise(model, noise_scale)

        optimizer.step()

    return model.state_dict()

# -----------------------------
# Server Aggregation (FedAvg)
# -----------------------------
def aggregate_models(global_model, client_updates):
    new_state = copy.deepcopy(global_model.state_dict())

    for key in new_state.keys():
        new_state[key] = torch.stack(
            [client_updates[i][key] for i in range(len(client_updates))]
        ).mean(0)

    global_model.load_state_dict(new_state)
    return global_model

# -----------------------------
# Simulated Data for Clients
# -----------------------------
def generate_client_data(num_clients=5, samples=100):
    client_data = []
    for _ in range(num_clients):
        X = torch.randn(samples, 10)
        y = torch.randint(0, 2, (samples,))
        client_data.append((X, y))
    return client_data

# -----------------------------
# Federated Training Loop
# -----------------------------
def federated_learning(rounds=5, num_clients=5):
    global_model = SimpleModel()
    clients = generate_client_data(num_clients)

    for r in range(rounds):
        print(f"Round {r+1}")

        client_updates = []
        for i, (data, targets) in enumerate(clients):
            updated_weights = train_client(
                global_model,
                data,
                targets,
                epochs=1,
                lr=0.01,
                max_norm=1.0,
                noise_scale=0.1   # DP noise
            )
            client_updates.append(updated_weights)

        global_model = aggregate_models(global_model, client_updates)

    return global_model

# -----------------------------
# Run
# -----------------------------
if __name__ == "__main__":
    model = federated_learning(rounds=10, num_clients=5)
    print("Training complete.")

Round 1
Round 2
Round 3
Round 4
Round 5
Round 6
Round 7
Round 8
Round 9
Round 10
Training complete.
